# DINOv3 for Plasma Segmentation - Exploration

This notebook explores using DINOv3 (facebook/dinov3-vits16) for object detection and segmentation on plasma data.

DINOv3 is the latest version (released August 2025) with **register tokens** for improved dense prediction tasks.

We'll start by loading a pretrained DINOv3 model and visualizing its feature extraction capabilities without training.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os

# Add ingestion_program to path (handle both relative and absolute paths)
notebook_dir = Path(os.getcwd())
ingestion_path = notebook_dir / 'ingestion_program'
solution_path = notebook_dir / 'solution'

# Add paths if they exist
if ingestion_path.exists():
    sys.path.insert(0, str(ingestion_path))
if solution_path.exists():
    sys.path.insert(0, str(solution_path))

from transformers import AutoModel, AutoImageProcessor
from tokam2d_utils import TokamDataset
from train_model_dinov3 import DINOv3Segmentation

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Load Pretrained DINOv2 Model

In [ ]:
# Load the pretrained DINOv3 backbone
print("Loading DINOv3 model...")
model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dinov3_backbone = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
dinov3_backbone.to(device)
dinov3_backbone.eval()

print(f"Model loaded on {device}")
print(f"Hidden size: {dinov3_backbone.config.hidden_size}")
print(f"Patch size: {dinov3_backbone.config.patch_size}")
print(f"Image size: {dinov3_backbone.config.image_size}")
print(f"Number of register tokens: {dinov3_backbone.config.num_register_tokens}")

## 3. Load Sample Data

Load a sample from the tokamak dataset to see how DINOv2 processes it.

In [ ]:
# Specify your data directory path here
# Example: data_dir = Path('../data/training')
data_dir = Path('./dev_phase/input_data/train')  # Adjust this path as needed

if data_dir.exists():
    dataset = TokamDataset(data_dir, include_unlabeled=True)
    print(f"Dataset loaded with {len(dataset)} samples")
    
    # Get a sample
    sample_image, sample_target = dataset[10]
    print(f"Image shape: {sample_image.shape}")
    print(f"Target keys: {sample_target.keys() if sample_target else 'None'}")
else:
    print(f"Data directory not found: {data_dir}")
    print("Creating synthetic sample for demonstration...")
    sample_image = torch.randn(1, 224, 224)
    sample_target = None

## 4. Visualize Input Image

In [ ]:
# Visualize the sample image
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
plt.title('Original Image')
plt.colorbar()

# Show normalized version (what DINOv2 will see)
plt.subplot(1, 2, 2)
normalized = F.interpolate(sample_image.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
plt.imshow(normalized.squeeze().cpu().numpy(), cmap='viridis')
plt.title('Normalized to 224x224')
plt.colorbar()

plt.tight_layout()
plt.show()

print(f"Image statistics:")
print(f"  Min: {sample_image.min():.4f}")
print(f"  Max: {sample_image.max():.4f}")
print(f"  Mean: {sample_image.mean():.4f}")
print(f"  Std: {sample_image.std():.4f}")

## 5. Extract DINOv2 Features

In [ ]:
# Prepare image for DINOv3 (expects 3-channel RGB)
# Plasma data is single-channel, so we convert it to 3 channels

# CONFIGURABLE: Change this to get more patches!
# 224x224 → 14x14 patches (196 total)
# 448x448 → 28x28 patches (784 total)
# 672x672 → 42x42 patches (1764 total)
input_size = 448  # Increased from 224 to get more patches

# Add batch dimension if needed
if sample_image.dim() == 2:  # (H, W)
    image_input = sample_image.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
elif sample_image.dim() == 3:  # (C, H, W)
    image_input = sample_image.unsqueeze(0)  # (1, C, H, W)
else:
    image_input = sample_image

# Resize to desired size
image_input = F.interpolate(
    image_input,
    size=(input_size, input_size),
    mode='bilinear',
    align_corners=False
)

# Convert single channel to 3 channels (grayscale to RGB)
if image_input.shape[1] == 1:
    image_input = image_input.repeat(1, 3, 1, 1)

image_input = image_input.to(device)

print(f"Input image shape: {image_input.shape} (using {input_size}x{input_size})")

# Extract features following HuggingFace documentation
batch_size, _, img_height, img_width = image_input.shape
patch_size = dinov3_backbone.config.patch_size
num_patches_height = img_height // patch_size
num_patches_width = img_width // patch_size
num_patches_flat = num_patches_height * num_patches_width

with torch.inference_mode():
    outputs = dinov3_backbone(image_input)
    last_hidden_states = outputs.last_hidden_state

print(f"\nFeature shape: {last_hidden_states.shape}")
print(f"  Expected: [{batch_size}, {1 + dinov3_backbone.config.num_register_tokens + num_patches_flat}, {dinov3_backbone.config.hidden_size}]")
print(f"  - 1 CLS token")
print(f"  - {dinov3_backbone.config.num_register_tokens} register tokens")
print(f"  - {num_patches_flat} patch tokens ({num_patches_height}x{num_patches_width})")
print(f"  - {dinov3_backbone.config.hidden_size} dimensional features")

# Extract tokens following HuggingFace pattern
cls_token = last_hidden_states[:, 0, :]
patch_features_flat = last_hidden_states[:, 1 + dinov3_backbone.config.num_register_tokens:, :]
patch_features = patch_features_flat.unflatten(1, (num_patches_height, num_patches_width))

print(f"\nCLS token shape: {cls_token.shape}")
print(f"Patch features shape: {patch_features.shape}")

# Reshape to (B, C, H, W) for visualization
spatial_features = patch_features.permute(0, 3, 1, 2)
patch_size_grid = num_patches_height  # Store for later use
print(f"Spatial features shape: {spatial_features.shape} (B, C, H, W)")
print(f"\n💡 Note: Using {input_size}x{input_size} gives us {num_patches_flat} patches!")
print(f"   Change 'input_size' variable above to adjust patch count")

## 5.1. Visualize Patch Token Embeddings

Each 16x16 patch gets its own 384-dimensional embedding. Let's visualize these local embeddings and see how they correspond to spatial locations in the image.

In [ ]:
# Visualize patch tokens structure
print("=" * 60)
print("PATCH TOKEN EMBEDDINGS - Local features for dense tasks")
print("=" * 60)

print(f"\n📍 Spatial Structure:")
print(f"   Image size: 224 x 224 pixels")
print(f"   Patch size: {dinov3_backbone.config.patch_size} x {dinov3_backbone.config.patch_size} pixels")
print(f"   Grid: {num_patches_height} x {num_patches_width} patches")
print(f"   Total patches: {num_patches_flat}")

print(f"\n🔢 Token Dimensions:")
print(f"   patch_features_flat shape: {patch_features_flat.shape}")
print(f"   → (batch=1, num_patches={num_patches_flat}, embedding_dim=384)")
print(f"\n   patch_features (unflattened) shape: {patch_features.shape}")
print(f"   → (batch=1, height={num_patches_height}, width={num_patches_width}, embedding_dim=384)")

print(f"\n🎯 Usage for Dense Prediction:")
print(f"   Each of the {num_patches_flat} patches has its own 384-D embedding")
print(f"   These local features preserve spatial structure")
print(f"   Perfect for pixel-wise tasks like segmentation!")

# Visualize individual patch embeddings
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Select 10 random patch locations
random_patches = np.random.choice(num_patches_flat, 10, replace=False)
patch_coords = [(idx // num_patches_width, idx % num_patches_width) for idx in random_patches]

for plot_idx, (h, w) in enumerate(patch_coords):
    ax = axes[plot_idx // 5, plot_idx % 5]
    
    # Get the 384-dimensional embedding for this patch
    patch_embedding = patch_features[0, h, w, :].cpu().numpy()  # (384,)
    
    # Visualize the embedding as a 1D signal
    ax.plot(patch_embedding, linewidth=0.5, alpha=0.7)
    ax.set_title(f'Patch [{h},{w}]', fontsize=10)
    ax.set_xlabel('Embedding dimension', fontsize=8)
    ax.set_ylabel('Value', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=7)
    
    # Add statistics
    stats_text = f'μ={patch_embedding.mean():.2f}\nσ={patch_embedding.std():.2f}'
    ax.text(0.95, 0.95, stats_text, transform=ax.transAxes, 
            fontsize=7, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Individual Patch Token Embeddings (384-D vectors)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show spatial correspondence
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare sample image for visualization (handle different input shapes)
sample_viz = sample_image.squeeze().cpu().numpy()  # Remove extra dims

# Original image
axes[0].imshow(sample_viz, cmap='viridis')
axes[0].set_title(f'Original Plasma Image\n({sample_viz.shape[0]}x{sample_viz.shape[1]} pixels)', 
                  fontsize=12, fontweight='bold')
axes[0].axis('off')

# Patch grid overlay (on 224x224 normalized version)
# Resize to 224x224 for grid overlay
if sample_image.dim() == 2:
    img_224 = sample_image.unsqueeze(0).unsqueeze(0)
elif sample_image.dim() == 3:
    img_224 = sample_image.unsqueeze(0)
else:
    img_224 = sample_image

img_224 = F.interpolate(img_224, size=(224, 224), mode='bilinear', align_corners=False)
img_224_viz = img_224.squeeze().cpu().numpy()

axes[1].imshow(img_224_viz, cmap='viridis', alpha=0.7)
# Draw grid lines
for i in range(num_patches_height + 1):
    axes[1].axhline(i * dinov3_backbone.config.patch_size - 0.5, color='red', linewidth=1, alpha=0.8)
for j in range(num_patches_width + 1):
    axes[1].axvline(j * dinov3_backbone.config.patch_size - 0.5, color='red', linewidth=1, alpha=0.8)
axes[1].set_title(f'Patch Grid\n({num_patches_height}x{num_patches_width} patches of 16x16 pixels)', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlim(-0.5, 224 - 0.5)
axes[1].set_ylim(224 - 0.5, -0.5)
axes[1].axis('off')

# Patch embeddings spatial layout (show embedding norms)
patch_norms = torch.norm(patch_features[0], dim=-1).cpu().numpy()  # (14, 14)
im = axes[2].imshow(patch_norms, cmap='plasma', interpolation='nearest')
axes[2].set_title('Patch Embedding Magnitudes\n(||embedding||₂ for each patch)', 
                  fontsize=12, fontweight='bold')
axes[2].set_xlabel('Patch X coordinate', fontsize=10)
axes[2].set_ylabel('Patch Y coordinate', fontsize=10)
plt.colorbar(im, ax=axes[2], label='L2 norm')

# Add grid
for i in range(num_patches_height + 1):
    axes[2].axhline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
for j in range(num_patches_width + 1):
    axes[2].axvline(j - 0.5, color='white', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ These {num_patches_flat} patch tokens (each 384-D) form the basis for dense prediction!")
print(f"✓ Register tokens keep global info separate, so these stay clean and spatially meaningful")

## 5.2. Patch Feature Statistics

Compute the mean feature value across all 384 dimensions for each patch to see which regions have higher activation.

In [ ]:
# Compute mean feature values across all channels for each patch
print("=" * 60)
print("PATCH FEATURE ANALYSIS")
print("=" * 60)

# Get patch embeddings: (num_patches, embedding_dim)
patch_embeddings = patch_features_flat[0]  # Remove batch dimension
print(f"\nPatch embeddings shape: {patch_embeddings.shape}")
print(f"  {num_patches_flat} patches, each with {dinov3_backbone.config.hidden_size}-D embedding")

# Compute mean across all 384 channels for each patch
patch_means = patch_embeddings.mean(dim=1)  # (num_patches,)
print(f"\nPatch means shape: {patch_means.shape}")
print(f"  Mean range: [{patch_means.min():.4f}, {patch_means.max():.4f}]")
print(f"  Overall mean: {patch_means.mean():.4f}")
print(f"  Std: {patch_means.std():.4f}")

# Reshape to spatial grid for visualization
patch_means_grid = patch_means.reshape(num_patches_height, num_patches_width).cpu().numpy()

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Original image for reference
ax = axes[0]
sample_viz = sample_image.squeeze().cpu().numpy()
ax.imshow(sample_viz, cmap='viridis')
ax.set_title('Original Plasma Image', fontweight='bold', fontsize=14)
ax.axis('off')

# 2. Mean feature values per patch
ax = axes[1]
im = ax.imshow(patch_means_grid, cmap='plasma', interpolation='nearest')
ax.set_title('Mean Feature Value per Patch\n(Average across 384 dimensions)', fontweight='bold', fontsize=14)
ax.set_xlabel('Patch X', fontsize=12)
ax.set_ylabel('Patch Y', fontsize=12)
plt.colorbar(im, ax=ax, label='Mean Feature Value')

# Add grid
for i in range(num_patches_height + 1):
    ax.axhline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
for j in range(num_patches_width + 1):
    ax.axvline(j - 0.5, color='white', linewidth=0.5, alpha=0.3)

# 3. Overlay on original image
ax = axes[2]
ax.imshow(sample_viz, cmap='gray', alpha=0.4)
im = ax.imshow(patch_means_grid, cmap='hot', alpha=0.6, interpolation='bilinear',
               extent=[0, sample_viz.shape[1], sample_viz.shape[0], 0])
ax.set_title('Mean Features Overlay\n(Hot = higher mean activation)', fontweight='bold', fontsize=14)
ax.axis('off')
plt.colorbar(im, ax=ax, label='Mean Feature Value')

plt.tight_layout()
plt.show()

print(f"\n✓ Mean feature values computed for {num_patches_flat} patches")
print(f"✓ Shows which regions have higher/lower average activation in the feature space")

## 6. Visualize Feature Maps

In [ ]:
# Visualize some feature channels
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Select 8 random feature channels to visualize
num_features = spatial_features.shape[1]
selected_channels = np.random.choice(num_features, 8, replace=False)

for idx, channel in enumerate(selected_channels):
    feature_map = spatial_features[0, channel].cpu().numpy()
    axes[idx].imshow(feature_map, cmap='viridis')
    axes[idx].set_title(f'Feature Channel {channel}')
    axes[idx].axis('off')

plt.suptitle('DINOv2 Feature Maps (Random Channels)', fontsize=16)
plt.tight_layout()
plt.show()

## 7. Visualize Attention Maps

DINOv2 uses self-attention. We can visualize attention patterns to see what the model focuses on.

In [ ]:
# Get attention weights from the model following HuggingFace pattern
# Note: DINOv3 model needs to explicitly request attentions
with torch.inference_mode():
    outputs = dinov3_backbone(image_input, output_attentions=True)

print(f"Available output keys: {outputs.keys()}")

# According to HF docs, attentions are returned when output_attentions=True
if hasattr(outputs, 'attentions') and outputs.attentions is not None:
    attentions = outputs.attentions
    print(f"\n✓ Attention weights available!")
    print(f"Number of layers: {len(attentions)}")
    print(f"Attention shape (last layer): {attentions[-1].shape}")
    print(f"  Format: (batch, num_heads, num_tokens, num_tokens)")
    
    # Get attention from CLS token to all patches in the last layer
    # Shape: (batch, num_heads, num_tokens, num_tokens)
    last_layer_attn = attentions[-1]  # Last layer
    
    # Extract CLS token attention to patches
    # Token order: [CLS, register_tokens..., patch_tokens...]
    num_register = dinov3_backbone.config.num_register_tokens
    
    # CLS attention to all patches (skip CLS and register tokens)
    cls_to_patches = last_layer_attn[0, :, 0, 1 + num_register:]  # (num_heads, num_patches)
    
    # Average over attention heads
    avg_attention = cls_to_patches.mean(dim=0)  # (num_patches,)
    
    # Reshape to 2D grid
    attention_map = avg_attention.reshape(patch_size_grid, patch_size_grid).cpu().numpy()
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    axes[0].set_title('Original Plasma Density')
    axes[0].axis('off')
    
    axes[1].imshow(attention_map, cmap='hot', interpolation='bilinear')
    axes[1].set_title('CLS Attention to Patches')
    axes[1].axis('off')
    
    # Overlay - properly handle sample_image dimensions
    if sample_image.dim() == 2:
        img_for_resize = sample_image.unsqueeze(0).unsqueeze(0)
    elif sample_image.dim() == 3:
        img_for_resize = sample_image.unsqueeze(0)
    else:
        img_for_resize = sample_image
    
    img_resized = F.interpolate(
        img_for_resize,
        size=(patch_size_grid, patch_size_grid),
        mode='bilinear',
        align_corners=False
    ).squeeze().cpu().numpy()
    
    axes[2].imshow(img_resized, cmap='gray', alpha=0.6)
    im = axes[2].imshow(attention_map, cmap='hot', alpha=0.4, interpolation='bilinear')
    axes[2].set_title('Attention Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️  Attention weights not returned by model")
    print("Using feature activation as alternative visualization\n")
    
    # Alternative: visualize feature norms
    feature_activation = torch.norm(spatial_features[0], dim=0).cpu().numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    axes[0].set_title('Original Plasma Density')
    axes[0].axis('off')
    
    axes[1].imshow(feature_activation, cmap='hot', interpolation='bilinear')
    axes[1].set_title('Feature Activation Map')
    axes[1].axis('off')
    
    # Overlay - properly handle sample_image dimensions
    if sample_image.dim() == 2:
        img_for_resize = sample_image.unsqueeze(0).unsqueeze(0)
    elif sample_image.dim() == 3:
        img_for_resize = sample_image.unsqueeze(0)
    else:
        img_for_resize = sample_image
    
    img_resized = F.interpolate(
        img_for_resize,
        size=feature_activation.shape,
        mode='bilinear',
        align_corners=False
    ).squeeze().cpu().numpy()
    
    axes[2].imshow(img_resized, cmap='gray', alpha=0.6)
    axes[2].imshow(feature_activation, cmap='hot', alpha=0.4, interpolation='bilinear')
    axes[2].set_title('Activation Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

## 8. Load Full DINOv2 Segmentation Model

Now let's load our custom DINOv2Segmentation model with detection and segmentation heads.

In [ ]:
# Load the full segmentation model
model = DINOv3Segmentation(num_classes=2, pretrained=True, model_name=model_name)
model.to(device)
model.eval()

print("DINOv3Segmentation model loaded")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 9. Test Object Detection (Pretrained)

Let's see how the model performs with randomly initialized detection heads (before training).

In [ ]:
# First, let's visualize the ground truth labels to see what we expect
print("Ground Truth Annotations:")
if sample_target is not None:
    print(f"Available keys: {sample_target.keys()}")
    
    if 'boxes' in sample_target:
        print(f"\n📦 Bounding Boxes: {sample_target['boxes'].shape}")
        print(f"   Number of objects: {len(sample_target['boxes'])}")
        if len(sample_target['boxes']) > 0:
            print(f"   Box format: (x, y, w, h)")
            for i, box in enumerate(sample_target['boxes']):
                print(f"   Box {i+1}: {box.cpu().numpy()}")
    
    if 'labels' in sample_target:
        print(f"\n🏷️  Labels: {sample_target['labels']}")
    
    if 'masks' in sample_target:
        print(f"\n🎭 Masks: {sample_target['masks'].shape}")
    
    # Visualize ground truth
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original image
    axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    axes[0].set_title('Original Plasma Image')
    axes[0].axis('off')
    
    # Image with ground truth bounding boxes
    axes[1].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
    if 'boxes' in sample_target and len(sample_target['boxes']) > 0:
        for i, box in enumerate(sample_target['boxes']):
            x, y, w, h = box.cpu().numpy()
            # Assuming box format is (x, y, w, h) - adjust if different
            rect = plt.Rectangle(
                (x, y), w, h,
                fill=False, edgecolor='lime', linewidth=2
            )
            axes[1].add_patch(rect)
            label_text = f"GT {i+1}"
            if 'labels' in sample_target:
                label_text += f" (L{sample_target['labels'][i].item()})"
            axes[1].text(
                x, y - 5,
                label_text,
                color='lime', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7)
            )
    axes[1].set_title('Ground Truth Bounding Boxes')
    axes[1].axis('off')
    
    # Ground truth mask
    if 'masks' in sample_target and len(sample_target['masks']) > 0:
        # Combine all masks
        combined_mask = sample_target['masks'].sum(dim=0).cpu().numpy()
        axes[2].imshow(sample_image.squeeze().cpu().numpy(), cmap='gray', alpha=0.5)
        axes[2].imshow(combined_mask, cmap='hot', alpha=0.5)
        axes[2].set_title('Ground Truth Masks Overlay')
    else:
        axes[2].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
        axes[2].text(0.5, 0.5, 'No mask annotations', 
                     ha='center', va='center', transform=axes[2].transAxes,
                     fontsize=12, color='white',
                     bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        axes[2].set_title('Ground Truth Masks')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("\n" + "="*60)
else:
    print("⚠️  No ground truth annotations available for this sample")
    print("   (This might be an unlabeled image)")

print("\nNow let's see what the untrained model predicts...\n")

In [ ]:
# Run inference
# Note: The model expects a list of images in (C, H, W) format

# Ensure sample_image is in the right format (C, H, W)
if sample_image.dim() == 2:  # (H, W)
    img_for_model = sample_image.unsqueeze(0)  # (1, H, W)
elif sample_image.dim() == 4:  # (B, C, H, W)
    img_for_model = sample_image.squeeze(0)  # (C, H, W)
else:
    img_for_model = sample_image  # Already (C, H, W)

print(f"Original input shape: {img_for_model.shape}")

# Convert grayscale to RGB by repeating the channel
if img_for_model.shape[0] == 1:
    img_for_model = img_for_model.repeat(3, 1, 1)
    print(f"Converted to 3-channel: {img_for_model.shape}")

with torch.no_grad():
    predictions = model([img_for_model.to(device)])

pred = predictions[0]
print(f"\nModel Predictions:")
print(f"  Boxes: {pred['boxes'].shape} - {len(pred['boxes'])} detections")
print(f"  Labels: {pred['labels'].shape}")
print(f"  Scores: {pred['scores'].shape}")

if len(pred['boxes']) > 0:
    print(f"\nDetected {len(pred['boxes'])} objects:")
    for i in range(min(5, len(pred['boxes']))):  # Show first 5
        box = pred['boxes'][i].cpu().numpy()
        print(f"  Object {i+1}:")
        print(f"    Label: {pred['labels'][i].item()}")
        print(f"    Score: {pred['scores'][i].item():.4f}")
        print(f"    Box (normalized [0,1]): [{box[0]:.4f}, {box[1]:.4f}, {box[2]:.4f}, {box[3]:.4f}]")
        print(f"    Box (pixels 512x512): [{box[0]*512:.1f}, {box[1]*512:.1f}, {box[2]*512:.1f}, {box[3]*512:.1f}]")
    if len(pred['boxes']) > 5:
        print(f"  ... and {len(pred['boxes']) - 5} more")
else:
    print("\n⚠️  No objects detected (expected for untrained model with random weights)")

## 10. Visualize Predictions

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original image
axes[0].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
axes[0].set_title('Original Image')
axes[0].axis('off')

# Image with bounding boxes
axes[1].imshow(sample_image.squeeze().cpu().numpy(), cmap='viridis')
if len(pred['boxes']) > 0:
    for box, label, score in zip(pred['boxes'], pred['labels'], pred['scores']):
        # Boxes are in normalized [0, 1] coordinates
        # Convert to pixel coordinates for visualization (512x512)
        x, y, w, h = box.cpu().numpy()
        x_pix, y_pix, w_pix, h_pix = x * 512, y * 512, w * 512, h * 512
        
        rect = plt.Rectangle(
            (x_pix, y_pix), w_pix, h_pix,
            fill=False, edgecolor='red', linewidth=2
        )
        axes[1].add_patch(rect)
        axes[1].text(
            x_pix, y_pix - 5,
            f'L{label.item()}: {score.item():.2f}',
            color='red', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
        )
axes[1].set_title('Predicted Bounding Boxes')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n💡 Note: Boxes are now in normalized [0, 1] coordinates")
print(f"   Multiply by 512 to get pixel coordinates in original image")

## 11. Feature Analysis

Let's analyze the features extracted by DINOv2 to understand what it captures.

In [ ]:
# Compute feature statistics
with torch.inference_mode():
    # Get features from backbone
    backbone_outputs = model.backbone(image_input)
    last_hidden_states = backbone_outputs.last_hidden_state
    
    # Get patch features (skip CLS and register tokens) following HF pattern
    num_register = model.num_register_tokens
    patch_features_flat = last_hidden_states[:, 1 + num_register:, :]
    
    # Compute statistics
    feature_norms = torch.norm(patch_features_flat, dim=-1)
    feature_mean = patch_features_flat.mean(dim=-1)
    feature_std = patch_features_flat.std(dim=-1)

# Visualize feature statistics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, data, title in zip(
    axes,
    [feature_norms, feature_mean, feature_std],
    ['Feature Norms', 'Feature Mean', 'Feature Std']
):
    stat_map = data.reshape(patch_size_grid, patch_size_grid).cpu().numpy()
    im = ax.imshow(stat_map, cmap='viridis')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print(f"\nFeature statistics:")
print(f"  Norm - min: {feature_norms.min():.4f}, max: {feature_norms.max():.4f}, mean: {feature_norms.mean():.4f}")
print(f"  Mean - min: {feature_mean.min():.4f}, max: {feature_mean.max():.4f}, mean: {feature_mean.mean():.4f}")
print(f"  Std - min: {feature_std.min():.4f}, max: {feature_std.max():.4f}, mean: {feature_std.mean():.4f}")

## 12. Summary

In this notebook, we explored:

1. ✅ Loading a pretrained DINOv3 model (facebook/dinov3-vits16-pretrain-lvd1689m)
2. ✅ Understanding DINOv3's register tokens for better dense predictions
3. ✅ Extracting and visualizing features from plasma images
4. ✅ Analyzing attention patterns from the vision transformer
5. ✅ Testing our custom DINOv3Segmentation model (untrained)
6. ✅ Visualizing predictions from randomly initialized heads

### Key DINOv3 Improvements over DINOv2:

- **Register tokens**: Dedicated memory slots for global information
- **Cleaner attention maps**: Reduced high-norm artifacts in patch tokens
- **Better dense prediction**: Improved performance on segmentation tasks
- **Released August 2025**: Latest state-of-the-art foundation model

### Next Steps:

- Train the model using `train_model()` function from `train_model_dinov3.py`
- Fine-tune the backbone by calling `model.unfreeze_backbone()`
- Experiment with different learning rates and training strategies
- Try larger models: `facebook/dinov3-vitb16-pretrain-lvd1689m` or `facebook/dinov3-vit7b16-pretrain-lvd1689m`
- Evaluate on validation data and visualize trained predictions

The pretrained DINOv3 backbone with register tokens provides even stronger visual features for plasma segmentation!

## 13. Train the DINOv3 Detection Model

Now let's train the model for 20 epochs with:
- **Data augmentation**: Random flips, rotations, translations, brightness
- **L1 loss**: For bounding box regression

In [ ]:
import time

print("=" * 70)
print("TRAINING DINOv3 DETECTION MODEL")
print("=" * 70)

# Training configuration
num_epochs = 20
batch_size = 2
learning_rate = 1e-4
augmentation_prob = 0.5  # Probability of applying augmentation

print(f"\n📋 Training Configuration:")
print(f"   Target epochs: {num_epochs}")
print(f"   Batch size: {batch_size}")
print(f"   Learning rate: {learning_rate}")
print(f"   Augmentation probability: {augmentation_prob}")
print(f"   Loss: L1 (for bounding box regression)")
print(f"   Device: {device}")
print(f"   Model: {model_name}")

# Load labeled training data (exclude unlabeled)
print(f"\n📁 Loading training data from: {data_dir}")
train_dataset = TokamDataset(data_dir, include_unlabeled=False)
print(f"   Total labeled samples: {len(train_dataset)}")

# Create custom training loop
from train_model_dinov3 import collate_fn, augment_image_and_boxes

train_dataloader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, collate_fn=collate_fn, shuffle=True
)

# Initialize fresh model for training
print(f"\n🔧 Initializing model...")
from train_model_dinov3 import DINOv3Segmentation
training_model = DINOv3Segmentation(num_classes=2, pretrained=True, model_name=model_name)
training_model.to(device)
training_model.train()

# Optimizer - only train the heads initially
trainable_params = [p for p in training_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=0.01)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=num_epochs * len(train_dataloader)
)

print(f"   Trainable parameters: {sum(p.numel() for p in trainable_params):,}")

# Start training
print(f"\n🚀 Starting training for {num_epochs} epochs with augmentation...\n")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    epoch_loss = 0
    epoch_cls_loss = 0
    epoch_bbox_loss = 0

    for batch_idx, (images, targets) in enumerate(train_dataloader):
        # Apply augmentation to images and boxes
        augmented_images = []
        augmented_targets = []
        
        for im, target in zip(images, targets):
            # Apply augmentation if target has boxes
            if target is not None and 'boxes' in target and target['boxes'] is not None and len(target['boxes']) > 0:
                aug_im, aug_boxes = augment_image_and_boxes(im, target['boxes'], p=augmentation_prob)
                augmented_images.append(aug_im)
                # Update target with augmented boxes
                aug_target = target.copy()
                aug_target['boxes'] = aug_boxes
                augmented_targets.append(aug_target)
            else:
                augmented_images.append(im)
                augmented_targets.append(target)
        
        # Convert single-channel to 3-channel RGB (DINOv3 expects 3 channels)
        augmented_images = [im.repeat(3, 1, 1) if im.shape[0] == 1 else im for im in augmented_images]
        augmented_images = [im.to(device) for im in augmented_images]
        augmented_targets = [
            {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()}
            for t in augmented_targets
        ]

        optimizer.zero_grad()
        loss_dict = training_model(augmented_images, augmented_targets)

        # Combine all losses
        total_loss = sum(loss for loss in loss_dict.values())
        total_loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

        optimizer.step()
        scheduler.step()

        epoch_loss += total_loss.item()
        epoch_cls_loss += loss_dict['loss_classifier'].item()
        epoch_bbox_loss += loss_dict['loss_box_reg'].item()

        if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == len(train_dataloader):
            print(f"  Batch {batch_idx+1}/{len(train_dataloader)}, "
                  f"Loss: {total_loss.item():.4f} "
                  f"(cls: {loss_dict['loss_classifier'].item():.3f}, "
                  f"bbox: {loss_dict['loss_box_reg'].item():.3f})")

    avg_loss = epoch_loss / len(train_dataloader)
    avg_cls = epoch_cls_loss / len(train_dataloader)
    avg_bbox = epoch_bbox_loss / len(train_dataloader)
    
    print(f"Epoch {epoch+1} completed. Avg losses - "
          f"Total: {avg_loss:.4f}, Cls: {avg_cls:.4f}, BBox: {avg_bbox:.4f}\n")

training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print(f"✓ Model trained for {num_epochs} epochs with augmentation")

# Move model to eval mode
training_model.eval()
trained_model = training_model

print("\n" + "=" * 70)

## 14. Evaluate Trained Model

Let's test the trained model on the same sample and compare with ground truth.

In [ ]:
# Get predictions from the trained model
print("🔍 Running inference with trained model...\n")

# Prepare image
if sample_image.dim() == 2:
    img_for_model = sample_image.unsqueeze(0)
elif sample_image.dim() == 4:
    img_for_model = sample_image.squeeze(0)
else:
    img_for_model = sample_image

# Convert to 3 channels if needed
if img_for_model.shape[0] == 1:
    img_for_model = img_for_model.repeat(3, 1, 1)

with torch.no_grad():
    trained_predictions = trained_model([img_for_model.to(device)])

trained_pred = trained_predictions[0]

print(f"Trained Model Predictions:")
print(f"  Boxes: {trained_pred['boxes'].shape} - {len(trained_pred['boxes'])} detections")
print(f"  Labels: {trained_pred['labels'].shape}")
print(f"  Scores: {trained_pred['scores'].shape}")

if len(trained_pred['boxes']) > 0:
    print(f"\n✓ Detected {len(trained_pred['boxes'])} objects:")
    for i in range(min(5, len(trained_pred['boxes']))):
        box = trained_pred['boxes'][i].cpu().numpy()
        print(f"   Object {i+1}:")
        print(f"      Label: {trained_pred['labels'][i].item()}")
        print(f"      Score: {trained_pred['scores'][i].item():.4f}")
        print(f"      Box (normalized [0,1]): [{box[0]:.4f}, {box[1]:.4f}, {box[2]:.4f}, {box[3]:.4f}]")
        print(f"      Box (pixels 512x512): [{box[0]*512:.1f}, {box[1]*512:.1f}, {box[2]*512:.1f}, {box[3]*512:.1f}]")
else:
    print("\n⚠️  No objects detected")

# Compare with ground truth
if sample_target is not None and 'boxes' in sample_target:
    print(f"\n📊 Ground Truth (in pixel coordinates):")
    print(f"   Number of objects: {len(sample_target['boxes'])}")
    for i, box in enumerate(sample_target['boxes'][:5]):
        box_np = box.cpu().numpy()
        print(f"   Object {i+1}:")
        print(f"      Box (pixels): [{box_np[0]:.1f}, {box_np[1]:.1f}, {box_np[2]:.1f}, {box_np[3]:.1f}]")
        print(f"      Box (normalized): [{box_np[0]/512:.4f}, {box_np[1]/512:.4f}, {box_np[2]/512:.4f}, {box_np[3]/512:.4f}]")

## 15. Visualize Training Results

Compare ground truth, untrained predictions, and trained predictions.

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(20, 8))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

sample_viz = sample_image.squeeze().cpu().numpy()

# Row 1: Bounding boxes comparison
# Ground Truth
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(sample_viz, cmap='viridis')
if sample_target is not None and 'boxes' in sample_target and len(sample_target['boxes']) > 0:
    for i, box in enumerate(sample_target['boxes']):
        x, y, w, h = box.cpu().numpy()
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2)
        ax1.add_patch(rect)
        ax1.text(x, y - 5, f"GT {i+1}", color='lime', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
ax1.set_title('Ground Truth Boxes', fontweight='bold', fontsize=12)
ax1.axis('off')

# Untrained predictions
ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(sample_viz, cmap='viridis')
if len(pred['boxes']) > 0:
    for i, (box, score) in enumerate(zip(pred['boxes'][:5], pred['scores'][:5])):
        # Convert normalized to pixels
        x, y, w, h = box.cpu().numpy()
        x_pix, y_pix, w_pix, h_pix = x * 512, y * 512, w * 512, h * 512
        rect = plt.Rectangle((x_pix, y_pix), w_pix, h_pix, fill=False, edgecolor='orange', linewidth=2)
        ax2.add_patch(rect)
        ax2.text(x_pix, y_pix - 5, f"{score.item():.2f}", color='orange', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
ax2.set_title(f'Untrained Model ({len(pred["boxes"])} detections)', fontweight='bold', fontsize=12)
ax2.axis('off')

# Trained predictions
ax3 = fig.add_subplot(gs[0, 2])
ax3.imshow(sample_viz, cmap='viridis')
if len(trained_pred['boxes']) > 0:
    for i, (box, score) in enumerate(zip(trained_pred['boxes'][:5], trained_pred['scores'][:5])):
        # Convert normalized to pixels
        x, y, w, h = box.cpu().numpy()
        x_pix, y_pix, w_pix, h_pix = x * 512, y * 512, w * 512, h * 512
        rect = plt.Rectangle((x_pix, y_pix), w_pix, h_pix, fill=False, edgecolor='red', linewidth=2)
        ax3.add_patch(rect)
        ax3.text(x_pix, y_pix - 5, f"{score.item():.2f}", color='red', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
ax3.set_title(f'Trained Model ({len(trained_pred["boxes"])} detections)', fontweight='bold', fontsize=12, color='red')
ax3.axis('off')

# Row 2: Score distributions and box coordinate comparison
# Score histogram comparison
ax4 = fig.add_subplot(gs[1, 0])
if len(pred['scores']) > 0 and len(trained_pred['scores']) > 0:
    ax4.hist(pred['scores'].cpu().numpy(), bins=20, alpha=0.5, label='Untrained', color='orange')
    ax4.hist(trained_pred['scores'].cpu().numpy(), bins=20, alpha=0.5, label='Trained', color='red')
    ax4.set_xlabel('Confidence Score', fontsize=12)
    ax4.set_ylabel('Count', fontsize=12)
    ax4.set_title('Detection Confidence Score Distribution', fontweight='bold', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Add statistics
    stats_text = f"Untrained: mean={pred['scores'].mean():.3f}, max={pred['scores'].max():.3f}\n"
    stats_text += f"Trained: mean={trained_pred['scores'].mean():.3f}, max={trained_pred['scores'].max():.3f}"
    ax4.text(0.02, 0.98, stats_text, transform=ax4.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
else:
    ax4.text(0.5, 0.5, 'No detections to compare', ha='center', va='center',
            fontsize=14, transform=ax4.transAxes)
    ax4.axis('off')

# Box coordinate comparison (normalized)
ax5 = fig.add_subplot(gs[1, 1])
if sample_target is not None and 'boxes' in sample_target and len(sample_target['boxes']) > 0:
    # Show normalized box values
    gt_boxes_norm = sample_target['boxes'].cpu().numpy() / 512.0
    ax5.text(0.5, 0.9, 'Ground Truth (normalized)', ha='center', transform=ax5.transAxes,
             fontweight='bold', fontsize=11, color='lime')
    
    text_str = ""
    for i, box in enumerate(gt_boxes_norm[:3]):
        text_str += f"Box {i+1}: [{box[0]:.3f}, {box[1]:.3f}, {box[2]:.3f}, {box[3]:.3f}]\n"
    ax5.text(0.1, 0.5, text_str, transform=ax5.transAxes, fontsize=10,
             verticalalignment='center', family='monospace')
    ax5.axis('off')
else:
    ax5.axis('off')

# Predicted boxes (normalized)
ax6 = fig.add_subplot(gs[1, 2])
if len(trained_pred['boxes']) > 0:
    pred_boxes_norm = trained_pred['boxes'].cpu().numpy()
    ax6.text(0.5, 0.9, 'Predictions (normalized)', ha='center', transform=ax6.transAxes,
             fontweight='bold', fontsize=11, color='red')
    
    text_str = ""
    for i, box in enumerate(pred_boxes_norm[:3]):
        text_str += f"Box {i+1}: [{box[0]:.3f}, {box[1]:.3f}, {box[2]:.3f}, {box[3]:.3f}]\n"
    ax6.text(0.1, 0.5, text_str, transform=ax6.transAxes, fontsize=10,
             verticalalignment='center', family='monospace')
    ax6.axis('off')
else:
    ax6.text(0.5, 0.5, 'No predictions', ha='center', va='center',
            fontsize=14, transform=ax6.transAxes)
    ax6.axis('off')

plt.suptitle(f'DINOv3 Training Results - {num_epochs} Epochs (Detection Only)', fontsize=16, fontweight='bold', y=0.995)
plt.show()

# Print summary
print("\n" + "=" * 70)
print("TRAINING RESULTS SUMMARY")
print("=" * 70)
print(f"\n📈 Detection Performance:")
print(f"   Untrained detections: {len(pred['boxes'])}")
print(f"   Trained detections: {len(trained_pred['boxes'])}")
print(f"   Ground truth objects: {len(sample_target['boxes']) if sample_target and 'boxes' in sample_target else 'N/A'}")

if len(trained_pred['scores']) > 0:
    print(f"\n📊 Confidence Scores (Trained Model):")
    print(f"   Mean: {trained_pred['scores'].mean():.4f}")
    print(f"   Max: {trained_pred['scores'].max():.4f}")
    print(f"   Min: {trained_pred['scores'].min():.4f}")

print("\n💡 All boxes are now in normalized [0, 1] coordinates")
print("   Multiply by 512 to convert to pixel coordinates")
print("\n✓ Visualization shows comparison between ground truth, untrained, and trained predictions")
print("=" * 70)

## 16. Validation Set Results

Now let's visualize predictions on the validation set, which includes samples without bounding boxes.

In [ ]:
# Import visualization function
from train_model_dinov3 import visualize_validation_results, split_dataset

# Get validation dataloader (we need to recreate it from the trained model)
# First, load the full dataset and split it
full_dataset_val = TokamDataset(data_dir, include_unlabeled=True)
train_indices, val_indices = split_dataset(full_dataset_val)

print(f"Creating validation visualization:")
print(f"  Total validation samples: {len(val_indices)}")
print(f"  Samples include both labeled and unlabeled images")

# Create validation subset and dataloader
val_dataset = torch.utils.data.Subset(full_dataset_val, val_indices)
from train_model_dinov3 import collate_fn
val_dataloader = torch.utils.data.DataLoader(
    val_dataset, batch_size=2, collate_fn=collate_fn, shuffle=False
)

# Visualize validation results
print("\n🎨 Generating validation visualizations...\n")
visualize_validation_results(
    trained_model, 
    val_dataloader, 
    full_dataset_val, 
    device=device, 
    num_samples=6
)

## 17. Compute Validation IoU Metrics

Calculate intersection over union (IoU) and other detection metrics on the validation set.

In [ ]:
# Compute IoU metrics on validation set
from train_model_dinov3 import compute_validation_iou

print("=" * 70)
print("VALIDATION SET IoU METRICS")
print("=" * 70)

print("\n🔍 Computing IoU metrics on validation set...")
iou_metrics = compute_validation_iou(trained_model, val_dataloader, device=device)

print(f"\n📊 Dataset Statistics:")
print(f"   Labeled samples: {iou_metrics['num_labeled']}")
print(f"   Unlabeled samples: {iou_metrics['num_unlabeled']}")
print(f"   Total validation samples: {iou_metrics['num_labeled'] + iou_metrics['num_unlabeled']}")

print(f"\n📈 Detection Performance:")
print(f"   True Positives: {iou_metrics['true_positives']}")
print(f"   False Positives: {iou_metrics['false_positives']}")
print(f"   False Negatives: {iou_metrics['false_negatives']}")

print(f"\n🎯 Metrics (IoU threshold = 0.5):")
print(f"   Mean IoU: {iou_metrics['mean_iou']:.4f}")
print(f"   Precision: {iou_metrics['precision']:.4f}")
print(f"   Recall: {iou_metrics['recall']:.4f}")
print(f"   F1 Score: {iou_metrics['f1_score']:.4f}")

print(f"\n💡 Interpretation:")
print(f"   - Mean IoU {iou_metrics['mean_iou']:.2%} overlap between predictions and ground truth")
print(f"   - Precision: {iou_metrics['precision']:.2%} of predictions are correct")
print(f"   - Recall: {iou_metrics['recall']:.2%} of ground truth boxes detected")
print(f"   - F1 Score: {iou_metrics['f1_score']:.2%} harmonic mean of precision and recall")

print("\n" + "=" * 70)

# Visualize metrics
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar chart of TP, FP, FN
ax1 = axes[0]
categories = ['True\nPositives', 'False\nPositives', 'False\nNegatives']
values = [iou_metrics['true_positives'], iou_metrics['false_positives'], iou_metrics['false_negatives']]
colors = ['green', 'orange', 'red']
bars = ax1.bar(categories, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Detection Counts', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
# Add value labels on bars
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Bar chart of metrics
ax2 = axes[1]
metrics_names = ['Precision', 'Recall', 'F1 Score']
metrics_values = [iou_metrics['precision'], iou_metrics['recall'], iou_metrics['f1_score']]
bars = ax2.bar(metrics_names, metrics_values, color=['blue', 'purple', 'cyan'], alpha=0.7, edgecolor='black', linewidth=2)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_ylim(0, 1)
ax2.set_title('Performance Metrics', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 threshold')
# Add value labels on bars
for bar, val in zip(bars, metrics_values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart of dataset composition
ax3 = axes[2]
dataset_labels = ['Labeled', 'Unlabeled']
dataset_values = [iou_metrics['num_labeled'], iou_metrics['num_unlabeled']]
colors_pie = ['lightgreen', 'lightcoral']
wedges, texts, autotexts = ax3.pie(dataset_values, labels=dataset_labels, autopct='%1.1f%%',
                                     colors=colors_pie, startangle=90, textprops={'fontsize': 12})
ax3.set_title('Validation Set Composition', fontsize=14, fontweight='bold')
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontweight('bold')

plt.suptitle(f'Validation Metrics - Mean IoU: {iou_metrics["mean_iou"]:.4f}', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Metrics computed and visualized")